# Forward Factor Backtester — End-to-End Runner

This notebook runs the full Forward Factor backtest in Google Colab. It clones the GitHub repo, installs dependencies, runs all 6 cells + ensemble + benchmarks, and renders the comparison dashboard.

**Before running:** You'll need a Polygon Options Advanced API key. Paste it into the secrets cell below. It's stored only in the notebook session — it is **NOT** committed to GitHub.

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/stegitforme/forward-factor-backtester.git
%cd forward-factor-backtester
!pip install -q -r requirements.txt

## 2. Configure secrets

Paste your Polygon API key below. Get it from https://massive.com/dashboard/keys.

In [ ]:
import os

# === EDIT THIS LINE ===
POLYGON_API_KEY = 'PASTE_YOUR_POLYGON_KEY_HERE'
# ======================

# Write secrets file (gitignored, won't be committed)
secrets_content = f'''
POLYGON_API_KEY = {POLYGON_API_KEY!r}
TRADIER_API_KEY = ''
GITHUB_PAT = ''
'''
with open('config/secrets.py', 'w') as f:
    f.write(secrets_content)

print('Secrets configured.')

## 3. Run unit tests (sanity check before backtest)

In [ ]:
!pytest tests/ -v --tb=short

## 4. Run benchmarks (SPY, QQQ, TQQQ/SGOV)

These run first because they're cheap and let us verify Polygon connectivity before the expensive FF backtest.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from src.data_layer import get_client
from src.benchmark import run_all_benchmarks

client = get_client()
benchmarks = run_all_benchmarks(client)

for name, result in benchmarks.items():
    print(f'{name:30s}  starting=${result.starting_equity:>12,.0f}  ending=${result.ending_equity:>12,.0f}')

## 5. Run the full Forward Factor backtest (6 cells + ensemble)

This is the expensive step — pulls option chains for all universe names across the full date range. Expect 30-60 minutes on the first run; subsequent runs use the disk cache and are much faster.

Note: the per-day candidate discovery in `find_candidates_for_day()` is currently a stub. To produce real trades, fill in the Polygon chain-resolution logic per the comments in `src/backtest.py`. Until then, the loop runs but produces empty trade logs (no signal-driven trades).

In [ ]:
from src.backtest import run_full_backtest
from src.earnings_filter import EarningsFilter

earnings = EarningsFilter(client)
result = run_full_backtest(client, earnings_filter=earnings)

print(f'\nCells run: {len(result.cell_results)}')
for name, cr in result.cell_results.items():
    print(f'  {name:25s}  final=${cr.final_equity:>12,.0f}  trades={len(cr.trade_log)}')

## 6. Compute metrics

In [ ]:
from src.metrics import compare_strategies, compute_metrics, correlation_matrix, compute_returns

# Build the strategy curves dict — FF cells + ensemble + benchmarks
strategy_curves = {name: cr.equity_curve for name, cr in result.cell_results.items()}
strategy_curves['ensemble'] = result.ensemble_curve
for name, br in benchmarks.items():
    strategy_curves[name] = br.equity_curve

# Trade logs (FF cells only — benchmarks don't have trade logs)
trade_logs = {name: cr.trade_log for name, cr in result.cell_results.items()}
trade_logs['ensemble'] = result.ensemble_trade_log

# Side-by-side comparison
comparison = compare_strategies(strategy_curves, trade_logs)
comparison.round(4)

In [ ]:
# Cross-cell correlation matrix (target: 0.4 - 0.85)
ff_returns = {name: compute_returns(cr.equity_curve)
              for name, cr in result.cell_results.items()
              if len(cr.equity_curve) > 1}
corr = correlation_matrix(ff_returns)
corr.round(2)

## 7. Render the dashboard

In [ ]:
from src.dashboard import render_dashboard
from src.metrics import compute_metrics

cell_metrics = {name: compute_metrics(cr.equity_curve, cr.trade_log)
                for name, cr in result.cell_results.items()}
ensemble_metrics = compute_metrics(result.ensemble_curve, result.ensemble_trade_log)

output_path = render_dashboard(
    strategy_curves=strategy_curves,
    cell_metrics=cell_metrics,
    ensemble_metrics=ensemble_metrics,
    correlation=corr,
    trade_logs=trade_logs,
)
print(f'Dashboard written to {output_path}')

# In Colab, display inline
from IPython.display import IFrame, HTML
with open(output_path) as f:
    html = f.read()
HTML(html)

## 8. Make the capital allocation decision

Look at the dashboard's **Capital Allocation Decision** panel. All checks must pass for allocation:

- ✅ Ensemble CAGR ≥ 15%
- ✅ Worst single cell Sharpe ≥ 1.0
- ✅ Win rate 50–70%
- ✅ Max DD ≤ 25%

**If all pass:** Allocate 10–15% of liquid net worth at quarter Kelly. Continue with paper trading for 1 quarter to verify live execution matches backtest.

**If any fail:** Document the reason, shelve the strategy, move on.